# ver11 Stage 1 - Foundation Model Adaptation

목표는 CNN 성능 경쟁이 아니라, Alzheimer MRI 분류에서 biomedical foundation model이 어디까지 전이되는지 확인하는 것입니다.

이 노트북은 `ver9/ver10`에서 정리한 환자 단위 5-Fold split과 Stage 1 문제 정의를 유지합니다.

Stage 1 문제:

- 0: `NonDemented`
- 1: `Demented = VeryMildDemented + MildDemented + ModerateDemented`

실험 흐름:

1. BiomedCLIP zero-shot
2. BiomedCLIP frozen image encoder + linear probe
3. BiomedCLIP frozen image encoder + adapter probe
4. 선택 사항: BiomedCLIP image encoder 일부 block partial fine-tuning

CNN EfficientNet 결과는 supervised baseline입니다. 이 ver11은 foundation model의 한계와 적응 효과를 설명하기 위한 비교 실험입니다.


## 1. 환경 점검 및 Seed 고정


In [1]:
import os
import sys
import time
import math
import gc
import re
import random
from pathlib import Path
from collections import defaultdict
from contextlib import contextmanager

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

SEED = 42


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python executable : {sys.executable}")
print(f"Python version    : {sys.version}")
print(f"torch version     : {torch.__version__}")
print(f"torch.version.cuda: {torch.version.cuda}")
print(f"cuda available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name          : {torch.cuda.get_device_name(0)}")
else:
    print("GPU name          : CUDA GPU not available")
print(f"device            : {device}")


Python executable : C:\Users\user\anaconda3\envs\alzheimer\python.exe
Python version    : 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
torch version     : 2.12.0.dev20260408+cu128
torch.version.cuda: 12.8
cuda available    : True
GPU name          : NVIDIA GeForce RTX 5070
device            : cuda


## 2. 실험 설정


In [2]:
DATA_ROOT = Path(r"C:\Users\user\Desktop\alzheimer_dataset\Data")
OUTPUT_DIR = Path(r"C:\Users\user\alzheimer\patient_level_stage1_foundation")
CACHE_DIR = OUTPUT_DIR / "cache"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

NEGATIVE_CLASSES = ["NonDemented"]
POSITIVE_CLASSES = ["VeryMildDemented", "MildDemented", "ModerateDemented"]
TASK_NAME = "non_vs_demented_biomedclip"

MODEL_NAME = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"

N_SPLITS = 5
INNER_VAL_RATIO = 0.15
FOLDS_TO_RUN = [1, 2, 3, 4, 5]

# 빠르게 foundation model 경향을 보려면 zero_shot, linear_probe, adapter_probe만 먼저 실행합니다.
EXPERIMENTS_TO_RUN = ["zero_shot", "linear_probe", "adapter_probe"]

# Partial fine-tuning은 시간이 더 걸립니다. 먼저 False로 두고, probe 결과를 본 뒤 True로 바꾸는 것을 권장합니다.
RUN_PARTIAL_FINETUNE = False
PARTIAL_FOLDS_TO_RUN = [1]
UNFREEZE_LAST_VISUAL_BLOCKS = 1

FEATURE_BATCH_SIZE = 128
PROBE_BATCH_SIZE = 256
FINETUNE_BATCH_SIZE = 16
NUM_WORKERS = 0
PIN_MEMORY = True
PERSISTENT_WORKERS = NUM_WORKERS > 0
USE_AMP = True

# Stage 1은 screening 성격이 강하므로 validation에서 민감도 우선 threshold를 고릅니다.
TARGET_VALIDATION_SENSITIVITY = 0.85
MIN_VALIDATION_SPECIFICITY = 0.65
THRESHOLD_GRID = np.round(np.arange(0.10, 0.7001, 0.025), 3)

PROBE_EPOCHS = 40
PROBE_LR = 1e-3
PROBE_WEIGHT_DECAY = 1e-4
PROBE_PATIENCE = 6
ADAPTER_HIDDEN_DIM = 128
ADAPTER_DROPOUT = 0.2

PARTIAL_EPOCHS = 6
PARTIAL_HEAD_LR = 5e-4
PARTIAL_VISUAL_LR = 1e-6
PARTIAL_WEIGHT_DECAY = 1e-4
PARTIAL_PATIENCE = 2

BALANCE_STRATEGY = "class_weight_sqrt"

FEATURE_CACHE_PATH = CACHE_DIR / f"{TASK_NAME}_all_original_features.pt"
RESULTS_PATH = OUTPUT_DIR / f"{TASK_NAME}_fold_results.csv"
SUMMARY_PATH = OUTPUT_DIR / f"{TASK_NAME}_summary.csv"

print(f"DATA_ROOT : {DATA_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"CACHE_DIR : {CACHE_DIR}")
print(f"MODEL_NAME: {MODEL_NAME}")
print(f"EXPERIMENTS_TO_RUN: {EXPERIMENTS_TO_RUN}")
print(f"RUN_PARTIAL_FINETUNE: {RUN_PARTIAL_FINETUNE}")
print(f"Threshold grid: {THRESHOLD_GRID.tolist()}")


DATA_ROOT : C:\Users\user\Desktop\alzheimer_dataset\Data
OUTPUT_DIR: C:\Users\user\alzheimer\patient_level_stage1_foundation
CACHE_DIR : C:\Users\user\alzheimer\patient_level_stage1_foundation\cache
MODEL_NAME: hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224
EXPERIMENTS_TO_RUN: ['zero_shot', 'linear_probe', 'adapter_probe']
RUN_PARTIAL_FINETUNE: False
Threshold grid: [0.1, 0.125, 0.15, 0.175, 0.2, 0.225, 0.25, 0.275, 0.3, 0.325, 0.35, 0.375, 0.4, 0.425, 0.45, 0.475, 0.5, 0.525, 0.55, 0.575, 0.6, 0.625, 0.65, 0.675, 0.7]


## 3. 환자 단위 Manifest 생성

이전 CNN 실험과 동일하게 파일명에서 `OAS1_xxxx` 환자 ID를 추출합니다.
같은 환자의 slice가 train/test에 동시에 들어가면 데이터 누수가 발생할 수 있으므로, 모든 split은 환자 ID 기준으로 진행합니다.


In [3]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PATIENT_PATTERN = re.compile(r"^(OAS1_\d+)", re.IGNORECASE)

rows = []
selected_classes = NEGATIVE_CLASSES + POSITIVE_CLASSES

for class_name in selected_classes:
    target = 0 if class_name in NEGATIVE_CLASSES else 1
    class_dir = DATA_ROOT / class_name
    assert class_dir.exists(), f"Class directory not found: {class_dir}"

    for image_path in sorted(class_dir.iterdir()):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        patient_match = PATIENT_PATTERN.match(image_path.name)
        assert patient_match, f"Patient ID를 추출할 수 없습니다: {image_path.name}"
        rows.append({
            "image_path": str(image_path),
            "class_name": class_name,
            "patient_id": patient_match.group(1).upper(),
            "target": target,
        })

manifest = pd.DataFrame(rows)
assert not manifest.empty
assert manifest.groupby("patient_id")["target"].nunique().max() == 1
assert manifest.groupby("patient_id")["class_name"].nunique().max() == 1

patient_table = (
    manifest.groupby("patient_id", as_index=False)
    .agg(
        target=("target", "first"),
        class_name=("class_name", "first"),
        image_count=("image_path", "count"),
    )
)

manifest_path = OUTPUT_DIR / f"{TASK_NAME}_all_images_manifest.csv"
patient_table_path = OUTPUT_DIR / f"{TASK_NAME}_patient_table.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
patient_table.to_csv(patient_table_path, index=False, encoding="utf-8-sig")

print(f"Images: {len(manifest):,}")
print(f"Patients: {len(patient_table):,}")
print("\n[Binary target patient counts]")
print(patient_table["target"].value_counts().sort_index())
print("\n[Original class patient counts]")
print(patient_table["class_name"].value_counts())
print(f"Manifest saved: {manifest_path}")


Images: 86,437
Patients: 347

[Binary target patient counts]
target
0    266
1     81
Name: count, dtype: int64

[Original class patient counts]
class_name
NonDemented         266
VeryMildDemented     58
MildDemented         21
ModerateDemented      2
Name: count, dtype: int64
Manifest saved: C:\Users\user\alzheimer\patient_level_stage1_foundation\non_vs_demented_biomedclip_all_images_manifest.csv


## 4. BiomedCLIP 모델 로드

BiomedCLIP은 biomedical image-text pretraining 기반 foundation model입니다.

이번 노트북에서 비교하려는 질문은 다음입니다.

- zero-shot만으로 Alzheimer MRI를 잘 구분하는가?
- image encoder를 고정하고 작은 classifier만 학습하면 좋아지는가?
- foundation model 전체가 아니라 작은 adapter 또는 일부 visual block만 조정하면 개선되는가?


In [4]:
import open_clip


def load_biomedclip_model():
    print("BiomedCLIP 로드 중...")
    model, preprocess = open_clip.create_model_from_pretrained(MODEL_NAME)
    tokenizer = open_clip.get_tokenizer(MODEL_NAME)
    model = model.to(device)
    model.eval()

    model_device = next(model.parameters()).device
    print(f"model parameter device: {model_device}")
    assert model_device.type == device.type, (
        f"BiomedCLIP model is not on expected device. expected={device}, current={model_device}"
    )
    return model, preprocess, tokenizer


biomed_model, preprocess, tokenizer = load_biomedclip_model()


C:\Users\user\anaconda3\envs\alzheimer\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BiomedCLIP 로드 중...


model parameter device: cuda:0


## 5. Feature extraction Dataset 및 Cache

Feature cache는 foundation model image encoder의 **고정된 representation**을 저장하는 단계입니다.

중요한 원칙:

- feature caching에는 augmentation을 넣지 않습니다.
- feature caching에는 sampler를 쓰지 않습니다.
- feature caching은 `shuffle=False`로 실행합니다.

이유:

- cache는 같은 이미지가 항상 같은 feature로 매핑된다는 전제가 있어야 합니다.
- augmentation을 cache에 섞으면 feature가 원본 representation인지 증강 representation인지 의미가 흐려집니다.
- class balancing은 feature를 뽑을 때가 아니라, 그 feature를 이용해 classifier를 학습할 때 적용해야 합니다.


In [5]:
class FoundationImageDataset(Dataset):
    def __init__(self, dataframe, preprocess):
        self.df = dataframe.reset_index(drop=True).copy()
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = Image.open(row["image_path"]).convert("RGB")
        image = self.preprocess(image)
        return (
            image,
            torch.tensor(int(row["target"]), dtype=torch.long),
            row["patient_id"],
            row["image_path"],
            row["class_name"],
        )


def extract_biomedclip_features(model, loader, total_count):
    model.eval()
    features = []
    labels = []
    patient_ids = []
    image_paths = []
    class_names = []

    start = time.time()
    seen = 0

    with torch.inference_mode():
        for batch_index, batch in enumerate(loader, start=1):
            images, batch_labels, batch_patient_ids, batch_image_paths, batch_class_names = batch
            images = images.to(device, non_blocking=True)

            with torch.amp.autocast(
                "cuda",
                enabled=(USE_AMP and torch.cuda.is_available()),
            ):
                batch_features = model.encode_image(images)
                batch_features = F.normalize(batch_features.float(), dim=-1)

            features.append(batch_features.cpu())
            labels.append(batch_labels.cpu())
            patient_ids.extend(list(batch_patient_ids))
            image_paths.extend(list(batch_image_paths))
            class_names.extend(list(batch_class_names))

            seen += images.size(0)
            elapsed = max(time.time() - start, 1e-6)
            speed = seen / elapsed
            eta_min = (total_count - seen) / max(speed, 1e-6) / 60
            if batch_index == 1 or batch_index % 20 == 0 or seen == total_count:
                print(
                    f"{seen:,}/{total_count:,} images | "
                    f"{speed:.1f} images/s | ETA {eta_min:.1f} min"
                )

    return {
        "features": torch.cat(features, dim=0),
        "labels": torch.cat(labels, dim=0),
        "patient_ids": patient_ids,
        "image_paths": image_paths,
        "class_names": class_names,
        "model_name": MODEL_NAME,
        "normalized": True,
    }


def load_or_create_feature_cache():
    if FEATURE_CACHE_PATH.exists():
        print(f"기존 feature cache 로드: {FEATURE_CACHE_PATH}")
        cached = torch.load(FEATURE_CACHE_PATH, map_location="cpu", weights_only=False)
    else:
        print("feature cache가 없어서 새로 생성합니다.")
        dataset = FoundationImageDataset(manifest, preprocess)
        loader = DataLoader(
            dataset,
            batch_size=FEATURE_BATCH_SIZE,
            shuffle=False,
            sampler=None,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            persistent_workers=PERSISTENT_WORKERS,
        )
        cached = extract_biomedclip_features(
            biomed_model,
            loader,
            total_count=len(dataset),
        )
        torch.save(cached, FEATURE_CACHE_PATH)
        print(f"feature cache 저장: {FEATURE_CACHE_PATH}")

    print(f"features shape: {tuple(cached['features'].shape)}")
    print(f"labels shape  : {tuple(cached['labels'].shape)}")
    print(f"patients      : {len(set(cached['patient_ids'])):,}")
    return cached


## 6. Feature cache 생성 또는 로드


In [6]:
cached_features = load_or_create_feature_cache()
feature_dim = int(cached_features["features"].shape[1])
print(f"feature_dim: {feature_dim}")


feature cache가 없어서 새로 생성합니다.
128/86,437 images | 137.9 images/s | ETA 10.4 min
2,560/86,437 images | 270.3 images/s | ETA 5.2 min
5,120/86,437 images | 276.5 images/s | ETA 4.9 min
7,680/86,437 images | 275.2 images/s | ETA 4.8 min
10,240/86,437 images | 266.9 images/s | ETA 4.8 min
12,800/86,437 images | 266.4 images/s | ETA 4.6 min
15,360/86,437 images | 269.5 images/s | ETA 4.4 min
17,920/86,437 images | 272.3 images/s | ETA 4.2 min
20,480/86,437 images | 273.0 images/s | ETA 4.0 min
23,040/86,437 images | 272.2 images/s | ETA 3.9 min
25,600/86,437 images | 271.1 images/s | ETA 3.7 min
28,160/86,437 images | 271.5 images/s | ETA 3.6 min
30,720/86,437 images | 272.0 images/s | ETA 3.4 min
33,280/86,437 images | 272.4 images/s | ETA 3.3 min
35,840/86,437 images | 272.7 images/s | ETA 3.1 min
38,400/86,437 images | 272.9 images/s | ETA 2.9 min
40,960/86,437 images | 273.1 images/s | ETA 2.8 min
43,520/86,437 images | 273.2 images/s | ETA 2.6 min
46,080/86,437 images | 273.2 images/s | 

## 7. 환자 단위 평가 함수와 threshold 선택 함수


In [7]:
def safe_auroc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)


def safe_auprc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, y_prob)


def aggregate_patient_predictions(patient_ids, labels, probabilities, threshold=0.5):
    grouped_probs = defaultdict(list)
    grouped_labels = {}
    for patient_id, label, prob in zip(patient_ids, labels, probabilities):
        grouped_probs[patient_id].append(float(prob))
        grouped_labels[patient_id] = int(label)

    ordered_ids = sorted(grouped_probs)
    patient_probs = np.array([np.mean(grouped_probs[pid]) for pid in ordered_ids])
    patient_labels = np.array([grouped_labels[pid] for pid in ordered_ids])
    patient_preds = (patient_probs >= threshold).astype(int)
    return ordered_ids, patient_labels, patient_probs, patient_preds


def calculate_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    specificity = cm[0, 0] / max(cm[0].sum(), 1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "auroc": safe_auroc(y_true, y_prob),
        "auprc": safe_auprc(y_true, y_prob),
    }


def build_threshold_table(y_true, y_prob, threshold_grid):
    rows = []
    for threshold in threshold_grid:
        metrics = calculate_metrics(y_true, y_prob, threshold=float(threshold))
        rows.append({
            "threshold": float(threshold),
            "distance_from_0_5": abs(float(threshold) - 0.5),
            **metrics,
        })
    return pd.DataFrame(rows)


def choose_sensitivity_first_threshold(
    y_true,
    y_prob,
    target_sensitivity=TARGET_VALIDATION_SENSITIVITY,
    min_specificity=MIN_VALIDATION_SPECIFICITY,
):
    table = build_threshold_table(y_true, y_prob, THRESHOLD_GRID)
    eligible = table[
        (table["sensitivity"] >= target_sensitivity)
        & (table["specificity"] >= min_specificity)
    ].copy()
    target_reached = not eligible.empty

    if not target_reached:
        eligible = table[table["specificity"] >= min_specificity].copy()
        if eligible.empty:
            eligible = table.copy()
        eligible = eligible[
            eligible["sensitivity"] == eligible["sensitivity"].max()
        ].copy()

    selected = eligible.sort_values(
        ["specificity", "precision", "distance_from_0_5"],
        ascending=[False, False, True],
    ).iloc[0]
    return float(selected["threshold"]), bool(target_reached), table


def make_class_weight_from_patients(patient_df):
    counts_series = patient_df["target"].value_counts().sort_index()
    counts = np.array(
        [counts_series.get(0, 0), counts_series.get(1, 0)],
        dtype=np.float64,
    )
    if BALANCE_STRATEGY == "class_weight_sqrt":
        values = 1.0 / np.sqrt(np.maximum(counts, 1))
        values /= values.mean()
        return torch.tensor(values, dtype=torch.float32, device=device)
    return None


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_total_parameters(model):
    return sum(p.numel() for p in model.parameters())


## 8. Zero-shot text prompt 정의

Zero-shot은 foundation model을 가장 직접적으로 사용하는 방식입니다.
다만 Alzheimer MRI는 미세한 구조 변화가 핵심이므로, zero-shot 성능이 낮게 나와도 이상한 결과가 아닙니다.
오히려 그 한계가 이후 linear probe와 adapter의 필요성을 설명해 줍니다.


In [8]:
NEGATIVE_PROMPTS = [
    "a brain MRI scan of a cognitively normal subject",
    "a normal brain MRI scan without dementia",
    "a brain MRI image of a non-demented patient",
]

POSITIVE_PROMPTS = [
    "a brain MRI scan of a patient with Alzheimer's disease",
    "a brain MRI image showing dementia",
    "a brain MRI scan of a demented patient",
]


def build_zero_shot_text_features(model, tokenizer):
    model.eval()
    prompt_groups = [NEGATIVE_PROMPTS, POSITIVE_PROMPTS]
    class_features = []

    with torch.inference_mode():
        for prompts in prompt_groups:
            tokens = tokenizer(prompts).to(device)
            with torch.amp.autocast(
                "cuda",
                enabled=(USE_AMP and torch.cuda.is_available()),
            ):
                text_features = model.encode_text(tokens)
                text_features = F.normalize(text_features.float(), dim=-1)
            class_feature = F.normalize(text_features.mean(dim=0, keepdim=True), dim=-1)
            class_features.append(class_feature)

    return torch.cat(class_features, dim=0).cpu()


zero_shot_text_features = build_zero_shot_text_features(biomed_model, tokenizer)
print(f"zero_shot_text_features shape: {tuple(zero_shot_text_features.shape)}")


zero_shot_text_features shape: (2, 512)


## 9. Cached feature 기반 dataset과 probe model


In [9]:
class CachedFeatureDataset(Dataset):
    def __init__(self, cached, indices):
        self.features = cached["features"]
        self.labels = cached["labels"].long()
        self.patient_ids = cached["patient_ids"]
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        index = int(self.indices[item])
        return (
            self.features[index].float(),
            self.labels[index],
            self.patient_ids[index],
        )


class LinearProbe(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.classifier = nn.Linear(input_dim, 2)

    def forward(self, features):
        return self.classifier(features)


class AdapterProbe(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.adapter = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
        )
        self.classifier = nn.Linear(input_dim, 2)

    def forward(self, features):
        normalized = self.norm(features)
        adapted = normalized + self.adapter(normalized)
        return self.classifier(adapted)


def make_probe_model(experiment_name, input_dim):
    if experiment_name == "linear_probe":
        return LinearProbe(input_dim).to(device)
    if experiment_name == "adapter_probe":
        return AdapterProbe(
            input_dim,
            hidden_dim=ADAPTER_HIDDEN_DIM,
            dropout=ADAPTER_DROPOUT,
        ).to(device)
    raise ValueError(experiment_name)


def indices_for_patient_ids(cached, patient_ids):
    patient_id_set = set(patient_ids)
    return [
        index
        for index, patient_id in enumerate(cached["patient_ids"])
        if patient_id in patient_id_set
    ]


def evaluate_probe_model(model, loader):
    model.eval()
    patient_ids, labels_all, probabilities_all = [], [], []

    with torch.inference_mode():
        for features, labels, batch_patient_ids in loader:
            features = features.to(device, non_blocking=True)
            with torch.amp.autocast(
                "cuda",
                enabled=(USE_AMP and torch.cuda.is_available()),
            ):
                logits = model(features)
                probs = logits.softmax(dim=1)[:, 1]

            patient_ids.extend(list(batch_patient_ids))
            labels_all.extend(labels.numpy().tolist())
            probabilities_all.extend(probs.detach().cpu().numpy().tolist())

    return aggregate_patient_predictions(
        patient_ids,
        labels_all,
        probabilities_all,
        threshold=0.5,
    )[:3]


def evaluate_zero_shot_on_patients(cached, patient_ids, text_features):
    indices = indices_for_patient_ids(cached, patient_ids)
    features = cached["features"][indices].float()
    labels = cached["labels"][indices].long().numpy()
    batch_patient_ids = [cached["patient_ids"][i] for i in indices]

    logits = features @ text_features.T
    probs = logits.softmax(dim=1)[:, 1].numpy()

    return aggregate_patient_predictions(
        batch_patient_ids,
        labels,
        probs,
        threshold=0.5,
    )[:3]


## 10. Fold 단위 학습 및 평가 함수

각 Fold에서는 다음 순서를 지킵니다.

1. outer train/test를 환자 단위로 분리
2. outer train 내부에서 inner train/validation 분리
3. validation에서 민감도 우선 threshold 선택
4. outer test에는 선택된 threshold만 적용

outer test를 threshold 선택에 사용하지 않기 때문에 평가 누수를 줄일 수 있습니다.


In [10]:
def train_probe_one_fold(
    experiment_name,
    fold_number,
    outer_train_patients,
    outer_test_patients,
):
    run_seed = SEED + fold_number * 100
    seed_everything(run_seed)

    inner_train_patients, inner_val_patients = train_test_split(
        outer_train_patients,
        test_size=INNER_VAL_RATIO,
        random_state=run_seed,
        stratify=outer_train_patients["target"],
    )

    train_indices = indices_for_patient_ids(
        cached_features,
        inner_train_patients["patient_id"],
    )
    val_indices = indices_for_patient_ids(
        cached_features,
        inner_val_patients["patient_id"],
    )
    test_indices = indices_for_patient_ids(
        cached_features,
        outer_test_patients["patient_id"],
    )

    train_dataset = CachedFeatureDataset(cached_features, train_indices)
    val_dataset = CachedFeatureDataset(cached_features, val_indices)
    test_dataset = CachedFeatureDataset(cached_features, test_indices)

    train_loader = DataLoader(
        train_dataset,
        batch_size=PROBE_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=PROBE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=PROBE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    model = make_probe_model(experiment_name, feature_dim)
    class_weight = make_class_weight_from_patients(inner_train_patients)
    criterion = nn.CrossEntropyLoss(weight=class_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=PROBE_LR,
        weight_decay=PROBE_WEIGHT_DECAY,
    )

    best_state = None
    best_epoch = 0
    best_val_auroc = -1.0
    epochs_without_improvement = 0

    print(
        f"\n[{experiment_name} fold {fold_number}] "
        f"inner train={len(inner_train_patients)}, "
        f"val={len(inner_val_patients)}, "
        f"outer test={len(outer_test_patients)}, "
        f"trainable={count_trainable_parameters(model):,}"
    )

    for epoch in range(1, PROBE_EPOCHS + 1):
        model.train()
        loss_sum, total = 0.0, 0

        for features, labels, _ in train_loader:
            features = features.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(features)
            loss = criterion(logits, labels)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * labels.size(0)
            total += labels.size(0)

        _, val_y_true, val_y_prob = evaluate_probe_model(model, val_loader)
        val_metrics = calculate_metrics(val_y_true, val_y_prob, threshold=0.5)

        if val_metrics["auroc"] > best_val_auroc + 1e-4:
            best_val_auroc = val_metrics["auroc"]
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"Epoch {epoch:02d}/{PROBE_EPOCHS} | "
            f"loss {loss_sum / max(total, 1):.4f} | "
            f"val AUROC {val_metrics['auroc']:.4f} "
            f"AUPRC {val_metrics['auprc']:.4f} | "
            f"early {epochs_without_improvement}/{PROBE_PATIENCE}"
        )

        if epochs_without_improvement >= PROBE_PATIENCE:
            print("Early stopping")
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    model = model.to(device)

    val_ids, val_y_true, val_y_prob = evaluate_probe_model(model, val_loader)
    selected_threshold, target_reached, _ = choose_sensitivity_first_threshold(
        val_y_true,
        val_y_prob,
    )
    test_ids, test_y_true, test_y_prob = evaluate_probe_model(model, test_loader)

    default_test = calculate_metrics(test_y_true, test_y_prob, threshold=0.5)
    calibrated_test = calculate_metrics(
        test_y_true,
        test_y_prob,
        threshold=selected_threshold,
    )

    checkpoint_path = (
        CHECKPOINT_DIR / f"{experiment_name}_fold{fold_number}.pt"
    )
    torch.save({
        "experiment": experiment_name,
        "fold": fold_number,
        "model_state_dict": best_state,
        "best_epoch": best_epoch,
        "best_val_auroc": best_val_auroc,
        "selected_threshold": selected_threshold,
        "validation_target_reached": target_reached,
        "feature_dim": feature_dim,
    }, checkpoint_path)

    result = {
        "experiment": experiment_name,
        "fold": fold_number,
        "best_epoch": best_epoch,
        "val_auroc": best_val_auroc,
        "selected_threshold": selected_threshold,
        "validation_target_reached": target_reached,
        "test_patients": len(test_ids),
        "default_sensitivity": default_test["sensitivity"],
        "default_specificity": default_test["specificity"],
        "default_f1": default_test["f1"],
        "calibrated_accuracy": calibrated_test["accuracy"],
        "calibrated_precision": calibrated_test["precision"],
        "calibrated_sensitivity": calibrated_test["sensitivity"],
        "calibrated_specificity": calibrated_test["specificity"],
        "calibrated_f1": calibrated_test["f1"],
        "calibrated_macro_f1": calibrated_test["macro_f1"],
        "auroc": calibrated_test["auroc"],
        "auprc": calibrated_test["auprc"],
        "trainable_params": count_trainable_parameters(model),
        "total_params": count_total_parameters(model),
    }

    del model, optimizer, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


def evaluate_zero_shot_one_fold(
    fold_number,
    outer_train_patients,
    outer_test_patients,
):
    run_seed = SEED + fold_number * 100
    _, inner_val_patients = train_test_split(
        outer_train_patients,
        test_size=INNER_VAL_RATIO,
        random_state=run_seed,
        stratify=outer_train_patients["target"],
    )

    val_ids, val_y_true, val_y_prob = evaluate_zero_shot_on_patients(
        cached_features,
        inner_val_patients["patient_id"],
        zero_shot_text_features,
    )
    selected_threshold, target_reached, _ = choose_sensitivity_first_threshold(
        val_y_true,
        val_y_prob,
    )
    test_ids, test_y_true, test_y_prob = evaluate_zero_shot_on_patients(
        cached_features,
        outer_test_patients["patient_id"],
        zero_shot_text_features,
    )

    default_test = calculate_metrics(test_y_true, test_y_prob, threshold=0.5)
    calibrated_test = calculate_metrics(
        test_y_true,
        test_y_prob,
        threshold=selected_threshold,
    )

    result = {
        "experiment": "zero_shot",
        "fold": fold_number,
        "best_epoch": 0,
        "val_auroc": calculate_metrics(val_y_true, val_y_prob, 0.5)["auroc"],
        "selected_threshold": selected_threshold,
        "validation_target_reached": target_reached,
        "test_patients": len(test_ids),
        "default_sensitivity": default_test["sensitivity"],
        "default_specificity": default_test["specificity"],
        "default_f1": default_test["f1"],
        "calibrated_accuracy": calibrated_test["accuracy"],
        "calibrated_precision": calibrated_test["precision"],
        "calibrated_sensitivity": calibrated_test["sensitivity"],
        "calibrated_specificity": calibrated_test["specificity"],
        "calibrated_f1": calibrated_test["f1"],
        "calibrated_macro_f1": calibrated_test["macro_f1"],
        "auroc": calibrated_test["auroc"],
        "auprc": calibrated_test["auprc"],
        "trainable_params": 0,
        "total_params": count_total_parameters(biomed_model),
    }
    return result


## 11. Foundation model 실험 실행

이 셀은 중간 결과 CSV가 있으면 이미 완료된 `(experiment, fold)` 조합을 건너뜁니다.
Windows 업데이트나 재부팅으로 중간에 끊겨도 이어서 실행할 수 있게 하기 위한 구조입니다.


In [11]:
outer_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)
patient_indices = np.arange(len(patient_table))
patient_targets = patient_table["target"].to_numpy()

if RESULTS_PATH.exists():
    results_df_existing = pd.read_csv(RESULTS_PATH)
    all_results = results_df_existing.to_dict("records")
    completed = {
        (str(row["experiment"]), int(row["fold"]))
        for row in all_results
    }
    print(f"기존 결과 {len(all_results)}개를 로드했습니다.")
    print(f"완료된 조합: {sorted(completed)}")
else:
    all_results = []
    completed = set()
    print("기존 결과가 없습니다. 새로 시작합니다.")

for fold_number, (outer_train_idx, outer_test_idx) in enumerate(
    outer_cv.split(patient_indices, patient_targets),
    start=1,
):
    if fold_number not in FOLDS_TO_RUN:
        continue

    outer_train_patients = patient_table.iloc[
        outer_train_idx
    ].reset_index(drop=True)
    outer_test_patients = patient_table.iloc[
        outer_test_idx
    ].reset_index(drop=True)

    print(
        f"\n===== FOUNDATION OUTER FOLD {fold_number} =====\n"
        f"train patients={len(outer_train_patients)}, "
        f"test patients={len(outer_test_patients)}\n"
        f"test target counts="
        f"{outer_test_patients['target'].value_counts().sort_index().to_dict()}"
    )

    for experiment_name in EXPERIMENTS_TO_RUN:
        key = (experiment_name, fold_number)
        if key in completed:
            print(f"[SKIP] 이미 완료됨: {experiment_name} fold {fold_number}")
            continue

        if experiment_name == "zero_shot":
            result = evaluate_zero_shot_one_fold(
                fold_number,
                outer_train_patients,
                outer_test_patients,
            )
        elif experiment_name in {"linear_probe", "adapter_probe"}:
            result = train_probe_one_fold(
                experiment_name,
                fold_number,
                outer_train_patients,
                outer_test_patients,
            )
        else:
            raise ValueError(experiment_name)

        print("Test:", result)
        all_results.append(result)
        completed.add(key)

        temp_path = RESULTS_PATH.with_suffix(".tmp.csv")
        pd.DataFrame(all_results).to_csv(
            temp_path,
            index=False,
            encoding="utf-8-sig",
        )
        temp_path.replace(RESULTS_PATH)
        print(f"중간 결과 저장: {RESULTS_PATH}")

results_df = pd.DataFrame(all_results)
display(results_df.sort_values(["experiment", "fold"]))


기존 결과가 없습니다. 새로 시작합니다.

===== FOUNDATION OUTER FOLD 1 =====
train patients=277, test patients=70
test target counts={0: 54, 1: 16}
Test: {'experiment': 'zero_shot', 'fold': 1, 'best_epoch': 0, 'val_auroc': 0.578125, 'selected_threshold': 0.5, 'validation_target_reached': False, 'test_patients': 70, 'default_sensitivity': 0.1875, 'default_specificity': np.float64(0.8148148148148148), 'default_f1': 0.20689655172413793, 'calibrated_accuracy': 0.6714285714285714, 'calibrated_precision': 0.23076923076923078, 'calibrated_sensitivity': 0.1875, 'calibrated_specificity': np.float64(0.8148148148148148), 'calibrated_f1': 0.20689655172413793, 'calibrated_macro_f1': 0.49984467225846535, 'auroc': 0.4537037037037037, 'auprc': 0.2317976142981973, 'trainable_params': 0, 'total_params': 195902721}
중간 결과 저장: C:\Users\user\alzheimer\patient_level_stage1_foundation\non_vs_demented_biomedclip_fold_results.csv

[linear_probe fold 1] inner train=235, val=42, outer test=70, trainable=1,026
Epoch 01/40 | loss 0

,experiment,fold,best_epoch,val_auroc,selected_threshold,validation_target_reached,test_patients,default_sensitivity,default_specificity,default_f1,calibrated_accuracy,calibrated_precision,calibrated_sensitivity,calibrated_specificity,calibrated_f1,calibrated_macro_f1,auroc,auprc,trainable_params,total_params
2,adapter_probe,1,9,0.896875,0.375,True,70,0.625000,0.870370,0.606061,0.842857,0.608696,0.875000,0.833333,0.717949,0.804519,0.907407,0.710235,133762,133762
5,adapter_probe,2,2,0.893750,0.400,True,70,0.941176,0.886792,0.820513,0.857143,0.629630,1.000000,0.811321,0.772727,0.834280,0.963374,0.837076,133762,133762
8,adapter_probe,3,8,0.896875,0.325,True,69,0.625000,0.886792,0.625000,0.826087,0.600000,0.750000,0.849057,0.666667,0.774510,0.867925,0.651108,133762,133762
11,adapter_probe,4,1,0.950000,0.425,True,69,0.500000,0.830189,0.484848,0.797101,0.541667,0.812500,0.792453,0.650000,0.753571,0.866745,0.656490,133762,133762
14,adapter_probe,5,1,0.953125,0.475,True,69,0.812500,0.830189,0.684211,0.869565,0.640000,1.000000,0.830189,0.780488,0.843852,0.899764,0.618640,133762,133762
1,linear_probe,1,2,0.900000,0.375,True,70,0.000000,1.000000,0.000000,0.785714,0.518519,0.875000,0.759259,0.651163,0.748262,0.929398,0.796117,1026,1026
4,linear_probe,2,4,0.896875,0.375,True,70,0.411765,0.962264,0.538462,0.857143,0.652174,0.882353,0.849057,0.750000,0.825000,0.936737,0.779885,1026,1026
7,linear_probe,3,1,0.868750,0.350,True,69,0.000000,1.000000,0.000000,0.768116,0.500000,0.687500,0.792453,0.578947,0.709474,0.865566,0.598419,1026,1026
10,linear_probe,4,3,0.937500,0.400,True,69,0.312500,0.943396,0.416667,0.797101,0.541667,0.812500,0.792453,0.650000,0.753571,0.867925,0.647342,1026,1026
13,linear_probe,5,31,0.965625,0.300,True,69,0.562500,0.905660,0.600000,0.797101,0.535714,0.937500,0.754717,0.681818,0.766441,0.890330,0.613416,1026,1026


## 12. 선택 사항: BiomedCLIP visual encoder partial fine-tuning

이 섹션은 시간이 더 걸리므로 기본값은 `RUN_PARTIAL_FINETUNE=False`입니다.

목적:

- foundation model 전체를 처음부터 재학습하지 않습니다.
- image encoder 대부분은 고정합니다.
- 마지막 visual block 일부와 classifier head만 학습합니다.

이 방식은 수업에서 말한 "전체 파라미터를 다 바꾸기보다, 도메인에 필요한 작은 부분만 조정한다"는 흐름과 연결됩니다.


In [12]:
class FineTuneImageDataset(Dataset):
    def __init__(self, dataframe, preprocess):
        self.df = dataframe.reset_index(drop=True).copy()
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = Image.open(row["image_path"]).convert("RGB")
        return (
            self.preprocess(image),
            torch.tensor(int(row["target"]), dtype=torch.long),
            row["patient_id"],
        )


class BiomedCLIPImageClassifier(nn.Module):
    def __init__(self, clip_model, input_dim):
        super().__init__()
        self.clip_model = clip_model
        self.classifier = nn.Linear(input_dim, 2)

    def forward(self, images):
        features = self.clip_model.encode_image(images)
        features = F.normalize(features.float(), dim=-1)
        return self.classifier(features)


def get_visual_block_container(clip_model):
    visual = getattr(clip_model, "visual", None)
    candidates = []
    if visual is not None:
        candidates.extend([
            ("visual.trunk.blocks", lambda: visual.trunk.blocks),
            ("visual.transformer.resblocks", lambda: visual.transformer.resblocks),
            ("visual.blocks", lambda: visual.blocks),
        ])

    for name, getter in candidates:
        try:
            blocks = getter()
            if hasattr(blocks, "__len__") and len(blocks) > 0:
                return name, blocks
        except Exception:
            pass
    return None, None


def configure_partial_finetune(classifier_model, n_last_blocks=1):
    for parameter in classifier_model.parameters():
        parameter.requires_grad = False

    for parameter in classifier_model.classifier.parameters():
        parameter.requires_grad = True

    name, blocks = get_visual_block_container(classifier_model.clip_model)
    if blocks is None:
        print("[WARN] visual block container를 찾지 못했습니다. classifier head만 학습합니다.")
        return "head_only"

    n_last_blocks = min(n_last_blocks, len(blocks))
    for block in list(blocks)[-n_last_blocks:]:
        for parameter in block.parameters():
            parameter.requires_grad = True

    print(f"Unfrozen visual blocks: {name}[-{n_last_blocks}:]")
    return f"{name}[-{n_last_blocks}:]"


def make_partial_optimizer(classifier_model):
    head_ids = {id(p) for p in classifier_model.classifier.parameters()}
    head_params, visual_params = [], []
    for parameter in classifier_model.parameters():
        if not parameter.requires_grad:
            continue
        if id(parameter) in head_ids:
            head_params.append(parameter)
        else:
            visual_params.append(parameter)

    groups = []
    if visual_params:
        groups.append({"params": visual_params, "lr": PARTIAL_VISUAL_LR})
    groups.append({"params": head_params, "lr": PARTIAL_HEAD_LR})
    return torch.optim.AdamW(groups, weight_decay=PARTIAL_WEIGHT_DECAY)


def evaluate_finetune_model(model, loader):
    model.eval()
    patient_ids, labels_all, probabilities_all = [], [], []
    with torch.inference_mode():
        for images, labels, batch_patient_ids in loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast(
                "cuda",
                enabled=(USE_AMP and torch.cuda.is_available()),
            ):
                logits = model(images)
                probs = logits.softmax(dim=1)[:, 1]
            patient_ids.extend(list(batch_patient_ids))
            labels_all.extend(labels.numpy().tolist())
            probabilities_all.extend(probs.detach().cpu().numpy().tolist())

    return aggregate_patient_predictions(
        patient_ids,
        labels_all,
        probabilities_all,
        threshold=0.5,
    )[:3]


In [13]:
def train_partial_finetune_one_fold(
    fold_number,
    outer_train_patients,
    outer_test_patients,
):
    run_seed = SEED + fold_number * 100
    seed_everything(run_seed)

    inner_train_patients, inner_val_patients = train_test_split(
        outer_train_patients,
        test_size=INNER_VAL_RATIO,
        random_state=run_seed,
        stratify=outer_train_patients["target"],
    )

    inner_train_ids = set(inner_train_patients["patient_id"])
    inner_val_ids = set(inner_val_patients["patient_id"])
    outer_test_ids = set(outer_test_patients["patient_id"])

    train_df = manifest[manifest["patient_id"].isin(inner_train_ids)].copy()
    val_df = manifest[manifest["patient_id"].isin(inner_val_ids)].copy()
    test_df = manifest[manifest["patient_id"].isin(outer_test_ids)].copy()

    train_loader = DataLoader(
        FineTuneImageDataset(train_df, preprocess),
        batch_size=FINETUNE_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        FineTuneImageDataset(val_df, preprocess),
        batch_size=FINETUNE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    test_loader = DataLoader(
        FineTuneImageDataset(test_df, preprocess),
        batch_size=FINETUNE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    fresh_clip_model, _, _ = load_biomedclip_model()
    model = BiomedCLIPImageClassifier(fresh_clip_model, feature_dim).to(device)
    trainable_scope = configure_partial_finetune(
        model,
        n_last_blocks=UNFREEZE_LAST_VISUAL_BLOCKS,
    )

    optimizer = make_partial_optimizer(model)
    class_weight = make_class_weight_from_patients(inner_train_patients)
    criterion = nn.CrossEntropyLoss(weight=class_weight)
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(USE_AMP and torch.cuda.is_available()),
    )

    best_state = None
    best_epoch = 0
    best_val_auroc = -1.0
    epochs_without_improvement = 0

    print(
        f"\n[partial_finetune fold {fold_number}] "
        f"scope={trainable_scope}, "
        f"trainable={count_trainable_parameters(model):,} / "
        f"{count_total_parameters(model):,}"
    )

    for epoch in range(1, PARTIAL_EPOCHS + 1):
        model.train()
        loss_sum, total = 0.0, 0
        for images, labels, _ in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(
                "cuda",
                enabled=(USE_AMP and torch.cuda.is_available()),
            ):
                logits = model(images)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_sum += loss.item() * labels.size(0)
            total += labels.size(0)

        _, val_y_true, val_y_prob = evaluate_finetune_model(model, val_loader)
        val_metrics = calculate_metrics(val_y_true, val_y_prob, threshold=0.5)

        if val_metrics["auroc"] > best_val_auroc + 1e-4:
            best_val_auroc = val_metrics["auroc"]
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"Epoch {epoch:02d}/{PARTIAL_EPOCHS} | "
            f"loss {loss_sum / max(total, 1):.4f} | "
            f"val AUROC {val_metrics['auroc']:.4f} "
            f"AUPRC {val_metrics['auprc']:.4f} | "
            f"early {epochs_without_improvement}/{PARTIAL_PATIENCE}"
        )

        if epochs_without_improvement >= PARTIAL_PATIENCE:
            print("Early stopping")
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    model = model.to(device)

    _, val_y_true, val_y_prob = evaluate_finetune_model(model, val_loader)
    selected_threshold, target_reached, _ = choose_sensitivity_first_threshold(
        val_y_true,
        val_y_prob,
    )
    test_ids, test_y_true, test_y_prob = evaluate_finetune_model(model, test_loader)

    default_test = calculate_metrics(test_y_true, test_y_prob, threshold=0.5)
    calibrated_test = calculate_metrics(
        test_y_true,
        test_y_prob,
        threshold=selected_threshold,
    )

    experiment_name = "partial_finetune"
    checkpoint_path = (
        CHECKPOINT_DIR / f"{experiment_name}_fold{fold_number}.pt"
    )
    torch.save({
        "experiment": experiment_name,
        "fold": fold_number,
        "model_state_dict": best_state,
        "best_epoch": best_epoch,
        "best_val_auroc": best_val_auroc,
        "selected_threshold": selected_threshold,
        "validation_target_reached": target_reached,
        "trainable_scope": trainable_scope,
    }, checkpoint_path)

    result = {
        "experiment": experiment_name,
        "fold": fold_number,
        "best_epoch": best_epoch,
        "val_auroc": best_val_auroc,
        "selected_threshold": selected_threshold,
        "validation_target_reached": target_reached,
        "test_patients": len(test_ids),
        "default_sensitivity": default_test["sensitivity"],
        "default_specificity": default_test["specificity"],
        "default_f1": default_test["f1"],
        "calibrated_accuracy": calibrated_test["accuracy"],
        "calibrated_precision": calibrated_test["precision"],
        "calibrated_sensitivity": calibrated_test["sensitivity"],
        "calibrated_specificity": calibrated_test["specificity"],
        "calibrated_f1": calibrated_test["f1"],
        "calibrated_macro_f1": calibrated_test["macro_f1"],
        "auroc": calibrated_test["auroc"],
        "auprc": calibrated_test["auprc"],
        "trainable_params": count_trainable_parameters(model),
        "total_params": count_total_parameters(model),
    }

    del model, optimizer, scaler, train_loader, val_loader, test_loader, fresh_clip_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


if RUN_PARTIAL_FINETUNE:
    if "all_results" not in globals():
        if RESULTS_PATH.exists():
            all_results = pd.read_csv(RESULTS_PATH).to_dict("records")
        else:
            all_results = []

    completed = {
        (str(row["experiment"]), int(row["fold"]))
        for row in all_results
    }

    outer_cv_partial = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED,
    )

    for fold_number, (outer_train_idx, outer_test_idx) in enumerate(
        outer_cv_partial.split(patient_indices, patient_targets),
        start=1,
    ):
        if fold_number not in PARTIAL_FOLDS_TO_RUN:
            continue
        key = ("partial_finetune", fold_number)
        if key in completed:
            print(f"[SKIP] 이미 완료됨: partial_finetune fold {fold_number}")
            continue

        outer_train_patients = patient_table.iloc[
            outer_train_idx
        ].reset_index(drop=True)
        outer_test_patients = patient_table.iloc[
            outer_test_idx
        ].reset_index(drop=True)

        result = train_partial_finetune_one_fold(
            fold_number,
            outer_train_patients,
            outer_test_patients,
        )
        print("Test:", result)
        all_results.append(result)

        temp_path = RESULTS_PATH.with_suffix(".tmp.csv")
        pd.DataFrame(all_results).to_csv(
            temp_path,
            index=False,
            encoding="utf-8-sig",
        )
        temp_path.replace(RESULTS_PATH)
        print(f"중간 결과 저장: {RESULTS_PATH}")
else:
    print("RUN_PARTIAL_FINETUNE=False 이므로 partial fine-tuning은 실행하지 않습니다.")
    print("probe 결과를 확인한 뒤 True로 바꾸고 PARTIAL_FOLDS_TO_RUN을 지정하세요.")


RUN_PARTIAL_FINETUNE=False 이므로 partial fine-tuning은 실행하지 않습니다.
probe 결과를 확인한 뒤 True로 바꾸고 PARTIAL_FOLDS_TO_RUN을 지정하세요.


## 13. Foundation model 결과 요약 및 CNN baseline 비교

이 표에서 봐야 할 핵심은 단순 accuracy가 아닙니다.

- zero-shot이 낮으면 foundation model의 직접 전이 한계를 의미합니다.
- linear probe가 개선되면 pretrained representation 자체는 쓸모가 있다는 뜻입니다.
- adapter probe가 더 좋아지면 작은 파라미터 적응의 효과를 보여줄 수 있습니다.
- partial fine-tuning이 좋아지면 도메인 적응이 필요했다는 결론을 강화합니다.


In [14]:
assert RESULTS_PATH.exists(), f"결과 CSV가 아직 없습니다: {RESULTS_PATH}"
results_df = pd.read_csv(RESULTS_PATH)

metric_columns = [
    "default_sensitivity",
    "default_specificity",
    "default_f1",
    "calibrated_accuracy",
    "calibrated_precision",
    "calibrated_sensitivity",
    "calibrated_specificity",
    "calibrated_f1",
    "calibrated_macro_f1",
    "auroc",
    "auprc",
    "selected_threshold",
    "trainable_params",
]

summary_rows = []
for experiment_name, group in results_df.groupby("experiment"):
    row = {
        "experiment": experiment_name,
        "folds": group["fold"].nunique(),
        "target_reached_folds": int(group["validation_target_reached"].sum()),
    }
    for metric in metric_columns:
        row[f"{metric}_mean"] = group[metric].mean()
        if group["fold"].nunique() > 1:
            row[f"{metric}_std"] = group[metric].std(ddof=1)
        else:
            row[f"{metric}_std"] = np.nan
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values("experiment")
summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")

display(results_df.sort_values(["experiment", "fold"]))
display(summary_df)
print(f"Summary saved: {SUMMARY_PATH}")

cnn_summary_path = Path(r"C:\Users\user\alzheimer\patient_level_stage1\sensitivity_first\sensitivity_target_summary.csv")
if cnn_summary_path.exists():
    cnn_summary = pd.read_csv(cnn_summary_path)
    print("\n[CNN EfficientNet ver10 sensitivity-first baseline]")
    display(cnn_summary)
else:
    print(f"CNN baseline summary not found: {cnn_summary_path}")


,experiment,fold,best_epoch,val_auroc,selected_threshold,validation_target_reached,test_patients,default_sensitivity,default_specificity,default_f1,calibrated_accuracy,calibrated_precision,calibrated_sensitivity,calibrated_specificity,calibrated_f1,calibrated_macro_f1,auroc,auprc,trainable_params,total_params
2,adapter_probe,1,9,0.896875,0.375,True,70,0.625000,0.870370,0.606061,0.842857,0.608696,0.875000,0.833333,0.717949,0.804519,0.907407,0.710235,133762,133762
5,adapter_probe,2,2,0.893750,0.400,True,70,0.941176,0.886792,0.820513,0.857143,0.629630,1.000000,0.811321,0.772727,0.834280,0.963374,0.837076,133762,133762
8,adapter_probe,3,8,0.896875,0.325,True,69,0.625000,0.886792,0.625000,0.826087,0.600000,0.750000,0.849057,0.666667,0.774510,0.867925,0.651108,133762,133762
11,adapter_probe,4,1,0.950000,0.425,True,69,0.500000,0.830189,0.484848,0.797101,0.541667,0.812500,0.792453,0.650000,0.753571,0.866745,0.656490,133762,133762
14,adapter_probe,5,1,0.953125,0.475,True,69,0.812500,0.830189,0.684211,0.869565,0.640000,1.000000,0.830189,0.780488,0.843852,0.899764,0.618640,133762,133762
1,linear_probe,1,2,0.900000,0.375,True,70,0.000000,1.000000,0.000000,0.785714,0.518519,0.875000,0.759259,0.651163,0.748262,0.929398,0.796117,1026,1026
4,linear_probe,2,4,0.896875,0.375,True,70,0.411765,0.962264,0.538462,0.857143,0.652174,0.882353,0.849057,0.750000,0.825000,0.936737,0.779885,1026,1026
7,linear_probe,3,1,0.868750,0.350,True,69,0.000000,1.000000,0.000000,0.768116,0.500000,0.687500,0.792453,0.578947,0.709474,0.865566,0.598419,1026,1026
10,linear_probe,4,3,0.937500,0.400,True,69,0.312500,0.943396,0.416667,0.797101,0.541667,0.812500,0.792453,0.650000,0.753571,0.867925,0.647342,1026,1026
13,linear_probe,5,31,0.965625,0.300,True,69,0.562500,0.905660,0.600000,0.797101,0.535714,0.937500,0.754717,0.681818,0.766441,0.890330,0.613416,1026,1026


,experiment,folds,target_reached_folds,default_sensitivity_mean,default_sensitivity_std,default_specificity_mean,default_specificity_std,default_f1_mean,default_f1_std,calibrated_accuracy_mean,...,calibrated_macro_f1_mean,calibrated_macro_f1_std,auroc_mean,auroc_std,auprc_mean,auprc_std,selected_threshold_mean,selected_threshold_std,trainable_params_mean,trainable_params_std
0,adapter_probe,5,5,0.700735,0.174692,0.860867,0.028796,0.644126,0.122376,0.838551,...,0.802147,0.038409,0.901043,0.039372,0.694710,0.086107,0.40,0.055902,133762.0,0.0
1,linear_probe,5,5,0.257353,0.251227,0.962264,0.040025,0.311026,0.291491,0.801035,...,0.760550,0.041824,0.897991,0.033547,0.687036,0.094032,0.36,0.037914,1026.0,0.0
2,zero_shot,5,0,0.222794,0.095997,0.778057,0.031984,0.225454,0.082200,0.648447,...,0.498839,0.048995,0.485927,0.072367,0.268011,0.066443,0.50,0.000000,0.0,0.0


Summary saved: C:\Users\user\alzheimer\patient_level_stage1_foundation\non_vs_demented_biomedclip_summary.csv

[CNN EfficientNet ver10 sensitivity-first baseline]


,target_sensitivity,target_reached_folds,median_threshold,mean_threshold,mean_test_sensitivity,std_test_sensitivity,mean_test_specificity,std_test_specificity,mean_test_precision,mean_test_f1,mean_test_macro_f1,mean_test_auroc,mean_test_auprc
0,0.80,5,0.475,0.455,0.738235,0.121752,0.887212,0.035311,0.667983,0.698486,0.800466,0.912233,0.733636
1,0.85,4,0.450,0.410,0.775735,0.114857,0.857233,0.045003,0.627273,0.690341,0.790372,0.912233,0.733636
2,0.90,4,0.450,0.410,0.775735,0.114857,0.857233,0.045003,0.627273,0.690341,0.790372,0.912233,0.733636


## 14. 결과 해석 가이드

보고서에서는 다음 원인-결과 구조로 설명하면 됩니다.

1. **Zero-shot**
   - 원인: BiomedCLIP은 biomedical image-text foundation model이지만 Alzheimer MRI severity는 미세한 구조 차이에 의존합니다.
   - 해석: zero-shot이 낮으면 foundation model의 직접 적용 한계를 보여줍니다.

2. **Frozen encoder + Linear probe**
   - 원인: image encoder의 representation은 유지하되, Alzheimer task에 맞는 decision boundary만 학습합니다.
   - 해석: zero-shot보다 좋아지면 pretrained feature는 유용하지만 task-specific classifier가 필요하다는 뜻입니다.

3. **Frozen encoder + Adapter probe**
   - 원인: 전체 foundation model을 건드리지 않고 작은 bottleneck adapter만 추가합니다.
   - 해석: linear probe보다 좋아지면 parameter-efficient adaptation의 근거가 됩니다.

4. **Partial fine-tuning**
   - 원인: MRI 도메인과 Alzheimer label에 맞게 visual encoder 마지막 일부 block만 조정합니다.
   - 해석: 성능이 개선되면 foundation model이 완전히 틀린 것이 아니라, 도메인 적응이 부족했던 것으로 설명할 수 있습니다.

5. **CNN baseline과 비교**
   - CNN이 더 좋다면: 작은 의료 데이터에서는 task-specific supervised CNN이 강력한 baseline임을 보여줍니다.
   - foundation model이 비슷하거나 더 좋다면: 적은 학습 파라미터로 사전학습 표현을 활용할 수 있음을 보여줍니다.

최종 메시지는 "foundation model이 무조건 좋다"가 아니라, "Alzheimer MRI에서는 zero-shot만으로는 부족했고, 환자 단위 검증과 sensitivity-first threshold를 적용한 뒤 domain adaptation 전략을 비교했다"가 되어야 합니다.
